# AI-Driven Audit & Compliance Validator
### Hackathon Challenge: Compliance Document Audit Agent on AMD Developer Cloud

This notebook implements an automated compliance validator using a multi-agent framework (**LangGraph** + **LangChain**). It ingests financial or insurance contracts, loads them into an in-memory vector database (**ChromaDB**), retrieves relevant context, and uses an LLM to evaluate the document against custom compliance rules. 

It features a two-step agentic validation loop:
1. **Auditor Agent**: Evaluates rule compliance and extracts verbatim evidence.
2. **Validator Agent (Self-Reflection)**: Verifies the quote's presence and accuracy to prevent hallucination, assigning a confidence score.

---  
## Step 1: Environment & GPU Check (AMD ROCm)

The code below verifies if **AMD ROCm** is active and PyTorch can access the AMD GPU accelerator.

In [2]:
import torch
import sys

print(f"Python Version: {sys.version}")
print(f"PyTorch Version: {torch.__version__}")
print(f"ROCm / CUDA GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Device Count: {torch.cuda.device_count()}")
else:
    print("No GPU detected. Model execution will default to CPU.")

ModuleNotFoundError: No module named 'torch'

---  
## Step 2: Install Dependencies

Run this cell to install the required libraries if they are not already installed on your cloud environment.

In [3]:
# Install missing dependencies in notebook container
!pip install -r requirements.txt

^C


  Using cached torch-2.12.0-cp313-cp313-win_amd64.whl.metadata (31 kB)
  Using cached transformers-5.12.0-py3-none-any.whl.metadata (33 kB)
  Using cached huggingface_hub-1.19.0-py3-none-any.whl.metadata (14 kB)
  Using cached accelerate-1.14.0-py3-none-any.whl.metadata (19 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached pyyaml-6.0.3-cp313-cp313-win_amd64.whl.metadata (2.4 kB)
  Using cached regex-2026.5.9-cp313-cp313-win_amd64.whl.metadata (41 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached typer-0.25.1-py3-none-any.whl.metadata (15 kB)
  Using cached anyio-4.13.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (

  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [19 lines of output]
      + c:\Users\Abhay\Desktop\AMD Hackathon\.venv\Scripts\python.exe C:\Users\Abhay\AppData\Local\Temp\pip-install-h32gu8rl\numpy_e32e3980ea5d4f78924054c607e4eaf1\vendored-meson\meson\meson.py setup C:\Users\Abhay\AppData\Local\Temp\pip-install-h32gu8rl\numpy_e32e3980ea5d4f78924054c607e4eaf1 C:\Users\Abhay\AppData\Local\Temp\pip-install-h32gu8rl\numpy_e32e3980ea5d4f78924054c607e4eaf1\.mesonpy-iy3dlzqk -Dbuildtype=release -Db_ndebug=if-release -Db_vscrt=md --native-file=C:\Users\Abhay\AppData\Local\Temp\pip-install-h32gu8rl\numpy_e32e3980ea5d4f78924054c607e4eaf1\.mesonpy-iy3dlzqk\meson-python-native-file.ini
      The Meson build system
      Version: 1.2.99
      Source dir: C:\Users\Abhay\AppData\Local\Temp\pip-install-h32gu8rl\numpy_e32e3980ea5d4f78924054c607e4eaf1
      Build dir: C:\Users\Abhay\AppData\Local\Temp\pip-install-h32gu8r

---  
## Step 3: Handle Document Loading (GitHub + Wget)

Since manual file upload may be disabled on your Jupyter server, you can upload your PDF or TXT documents to a public GitHub repository and download them directly into the workspace using the `wget` or `requests` helper below.

In [ ]:
import os
import requests

def download_from_github(raw_url: str, save_name: str):
    """Downloads a file from a URL (e.g. raw github user content) to local disk."""
    print(f"Downloading {raw_url}...")
    response = requests.get(raw_url, stream=True)
    if response.status_code == 200:
        with open(save_name, 'wb') as f:
            for chunk in response.iter_content(chunk_size=1024):
                f.write(chunk)
        print(f"✔ Successfully saved as: {save_name}")
    else:
        print(f"❌ Failed to download. Status code: {response.status_code}")

# EXAMPLE USAGE (Uncomment and replace with your URL if needed):
# download_from_github("https://raw.githubusercontent.com/username/repo/main/my_doc.pdf", "contract.pdf")

---  
## Step 4: Generate Local Sample Data

To test the validator immediately without downloading custom files, we generate local sample data (`sample_contract.txt` containing violations, and `rules.json` defining standard lending rules).

In [1]:
# Generate mock rules and mock policy text locally
!python generate_samples.py

Created sample rules at: sample_data\rules.json
Created sample contract at: sample_data\sample_contract.txt


---  
## Step 5: Initialize RAG and Load Rules

Let's load the generated compliance rules and document content.

In [ ]:
import json

# Load rules
rules_path = "sample_data/rules.json"
with open(rules_path, "r", encoding="utf-8") as f:
    rules = json.load(f)
    
# Load contract document
doc_path = "sample_data/sample_contract.txt"
with open(doc_path, "r", encoding="utf-8") as f:
    document_text = f.read()

print(f"✔ Loaded {len(rules)} compliance rules.")
print(f"✔ Loaded document '{doc_path}' ({len(document_text)} characters).")

---  
## Step 6: Configure and Run the LangGraph Agent

Configure your local Hugging Face model execution on the AMD GPU. Since we are running on an AMD Instinct MI300X, VRAM is not an issue, so you can choose to use different models for the Auditor and the Validator agents to reduce bias and maximize accuracy (e.g., Qwen-7B for Auditor and Llama-3-8B or Qwen-14B/32B for Validator).

In [ ]:
from compliance_agent import run_compliance_audit

# Model Settings (MI300X can easily run different large models for Auditor and Validator)
MODEL_NAME = "Qwen/Qwen2.5-14B-Instruct"      # Model for the Auditor node
VALIDATOR_NAME = "Qwen/Qwen2.5-7B-Instruct"   # Model for the Validator node

print(f"Executing audit with Auditor: '{MODEL_NAME}' and Validator: '{VALIDATOR_NAME}'...")
audit_reports = run_compliance_audit(
    document_text=document_text,
    rules=rules,
    model_name=MODEL_NAME,
    validator_name=VALIDATOR_NAME
)
print("✔ Audit Execution Completed.")

---  
## Step 7: View Interactive Report Dashboard

Now we render the compliance results table and structured audit metrics.

In [ ]:
import pandas as pd
from IPython.display import display, HTML, Markdown

# 1. Print Summary Statistics
total = len(audit_reports)
passes = sum(1 for r in audit_reports if r["status"] == "PASS")
fails = sum(1 for r in audit_reports if r["status"] == "FAIL")
warnings = sum(1 for r in audit_reports if r["status"] == "WARNING")
nas = sum(1 for r in audit_reports if r["status"] == "N/A")

summary_html = f"""
<div style="display: flex; gap: 15px; justify-content: space-around; margin: 20px 0; font-family: sans-serif;">
    <div style="background-color: #e2f0d9; padding: 15px 25px; border-radius: 8px; text-align: center;">
        <div style="font-size: 1.8em; font-weight: bold; color: #385723;">{passes}</div>
        <div style="color: #385723; font-weight: bold;">PASS ✅</div>
    </div>
    <div style="background-color: #fce4d6; padding: 15px 25px; border-radius: 8px; text-align: center;">
        <div style="font-size: 1.8em; font-weight: bold; color: #c65911;">{fails}</div>
        <div style="color: #c65911; font-weight: bold;">FAIL ❌</div>
    </div>
    <div style="background-color: #fff2cc; padding: 15px 25px; border-radius: 8px; text-align: center;">
        <div style="font-size: 1.8em; font-weight: bold; color: #7f6000;">{warnings}</div>
        <div style="color: #7f6000; font-weight: bold;">WARNING ⚠️</div>
    </div>
    <div style="background-color: #ededed; padding: 15px 25px; border-radius: 8px; text-align: center;">
        <div style="font-size: 1.8em; font-weight: bold; color: #595959;">{nas}</div>
        <div style="color: #595959;">N/A ➖</div>
    </div>
</div>
"""
display(Markdown("# Executive Audit Summary"))
display(HTML(summary_html))

# 2. Build Pandas DataFrame Matrix
df_data = []
for r in audit_reports:
    df_data.append({
        "Rule ID": r["rule_id"],
        "Rule Description": r["rule_name"],
        "Category": r["category"],
        "Status": r["status"],
        "Confidence": f"{r['confidence_score']}%",
        "Verbatim Quote": r["evidence"] if r["evidence"] else "N/A"
    })
df = pd.DataFrame(df_data)

# Custom CSS coloring
def color_status(val):
    if val == 'PASS':
        return 'background-color: #d5e8d4; color: #274e13; font-weight: bold;'
    elif val == 'FAIL':
        return 'background-color: #f8cecc; color: #660000; font-weight: bold;'
    elif val == 'WARNING':
        return 'background-color: #fff2cc; color: #7f6000; font-weight: bold;'
    return 'background-color: #f5f5f5; color: #666666;'

styled_df = df.style.applymap(color_status, subset=['Status'])
display(styled_df)

---  
## Step 8: Detailed Audit Log with Quotations

Below is the comprehensive explanation log, showing the reasoning and original contract quotes for each compliance rule audit.

In [ ]:
for i, r in enumerate(audit_reports):
    status = r["status"]
    color = "#385723" if status == "PASS" else "#c65911" if status == "FAIL" else "#7f6000" if status == "WARNING" else "#595959"
    bg_color = "#e2f0d9" if status == "PASS" else "#fce4d6" if status == "FAIL" else "#fff2cc" if status == "WARNING" else "#f2f2f2"
    icon = "✅ PASS" if status == "PASS" else "❌ FAIL" if status == "FAIL" else "⚠️ WARNING" if status == "WARNING" else "➖ N/A"
    
    html_card = f"""
    <div style="border-left: 6px solid {color}; background-color: {bg_color}; padding: 15px; margin: 18px 0; border-radius: 4px; font-family: Arial, sans-serif;">
        <h3 style="margin-top: 0; color: #333;">{i+1}. {r['rule_name']} <span style="font-size: 0.8em; color: #666;">({r['rule_id']})</span></h3>
        <p style="margin: 4px 0;"><strong>Category:</strong> {r['category']}</p>
        <p style="margin: 4px 0;"><strong>Requirement:</strong> {r['description']}</p>
        <p style="margin: 6px 0; font-size: 1.05em;"><strong>Status:</strong> <span style="color: {color}; font-weight: bold;">{icon}</span> (Confidence Score: {r['confidence_score']}% - Evidence Verbatim: {r['is_evidence_verbatim']})</p>
        <p style="margin: 6px 0; line-height: 1.4;"><strong>Reasoning:</strong> {r['reason']}</p>
        {f'<div style="background-color: rgba(255,255,255,0.75); border-left: 3px solid {color}; padding: 8px 12px; margin: 8px 0; font-style: italic;">"{r["evidence"]}"</div>' if r["evidence"] else ''}
        <p style="margin: 4px 0; font-size: 0.9em; color: #555; border-top: 1px dashed #ccc; padding-top: 6px;"><strong>Validator Review Details:</strong> {r['validation_notes']}</p>
    </div>
    """
    display(HTML(html_card))

---  
## Step 9: Save Audit Log Files

Exports the run findings as `audit_report.json` and a clean `audit_report.md` Markdown report.

In [ ]:
from main import save_reports

save_reports(audit_reports, "audit_report", "sample_contract.txt")
print("✔ Reports exported successfully.")